# Programming Assignment: Imputation Impact Assessment

Welcome to the **Imputation Impact Assessment** assignment!

Choosing an imputation strategy is not just about filling missing cells — it's about preserving the statistical properties that downstream models rely on. The theory introduces two key evaluation tools:

- **Kolmogorov–Smirnov test** — checks whether the imputed distribution is statistically distinguishable from the observed distribution. A high p-value ($\gg 0.05$) and low KS statistic ($< 0.15$) indicate the imputation preserved distributional shape.
- **Frobenius norm** on the correlation matrix — $\|R^{after} - R^{before}\|_F = \sqrt{\sum_{ij}(r^{after}_{ij} - r^{before}_{ij})^2}$. Near 0 means pairwise relationships are intact; above 1.50 signals severe distortion.

**What You Will Do in This Assignment:**

* Establish a baseline model trained on complete data
* Measure how listwise deletion affects model performance
* Compare the impact of simple vs. advanced imputation on model metrics
* Build a comprehensive impact report across all strategies
* Evaluate distributional fidelity using KS tests and Frobenius norm

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* **Avoid using global variables** unless absolutely necessary. The grader tests your code in an isolated environment.

---

## Table of Contents
- [Imports](#imports)
- [1 - Data Preparation](#1---data-preparation)
    - [1.1 - Generating Complete Dataset](#11---generating-complete-dataset)
    - [1.2 - Introducing Missing Values](#12---introducing-missing-values)
- [2 - Baseline Model](#2---baseline-model)
    - **[Exercise 1 - `prepare_baseline_model`](#exercise-1---prepare_baseline_model)**
- [3 - Deletion Impact](#3---deletion-impact)
    - **[Exercise 2 - `apply_listwise_deletion`](#exercise-2---apply_listwise_deletion)**
- [4 - Simple Imputation Impact](#4---simple-imputation-impact)
    - **[Exercise 3 - `apply_simple_imputation`](#exercise-3---apply_simple_imputation)**
- [5 - Advanced Imputation Impact](#5---advanced-imputation-impact)
    - **[Exercise 4 - `apply_advanced_imputation`](#exercise-4---apply_advanced_imputation)**
- [6 - Comprehensive Impact Report](#6---comprehensive-impact-report)
    - **[Exercise 5 - `create_impact_report`](#exercise-5---create_impact_report)**

*Note: Exercises 3–5 take a **single** dataset (not train/test split) and the test harness handles the splitting internally.*

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

In [ ]:
import helper_utils
import unittests

<a name='1---data-preparation'></a>
## 1 - Data Preparation

<a name='11---generating-complete-dataset'></a>
### 1.1 - Generating Complete Dataset

We generate a synthetic dataset with known relationships between features and target. This serves as our **ground truth** — the baseline against which we measure the impact of different missing-data handling strategies.

In [ ]:
# Generate complete dataset (no missing values)
X_complete, y = helper_utils.generate_complete_dataset(n_samples=500, n_features=5, random_state=42)

print(f"Dataset shape: {X_complete.shape}")
print(f"Missing values: {X_complete.isnull().sum().sum()}")
print(f"\nFeature statistics:")
X_complete.describe().round(2)

<a name='12---introducing-missing-values'></a>
### 1.2 - Introducing Missing Values

We introduce MCAR missingness (20% rate). Under MCAR, the ground truth relationship between features and target is preserved in the observed data — the bias risk is low but we still lose statistical power.

In [ ]:
X_missing = helper_utils.introduce_missing_values(X_complete, missing_rate=0.2, random_state=42)

print("Missing values per feature:")
print(X_missing.isnull().sum())
print(f"\nTotal missing: {X_missing.isnull().sum().sum()}")
print(f"Missing %:     {X_missing.isnull().sum().sum() / X_missing.size * 100:.1f}%")

In [ ]:
helper_utils.plot_missing_pattern(X_missing)

<a name='2---baseline-model'></a>
## 2 - Baseline Model

Before evaluating imputation strategies, we train a model on **complete data** (no missing values). This is the "best case" performance ceiling — the maximum we can expect any imputation strategy to approach.

<a name='exercise-1---prepare_baseline_model'></a>
### **Exercise 1 - `prepare_baseline_model`**

**Your Task:**

Implement `prepare_baseline_model(X_train, X_test, y_train, y_test)` that trains a `LinearRegression` on complete data and returns performance metrics.

**Requirements:**
- Train a `LinearRegression` model
- Return a `dict` with keys `'rmse'`, `'mae'`, `'r2'`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** `model = LinearRegression()` then `model.fit(X_train, y_train)`

**Step 2:** `y_pred = model.predict(X_test)`

**Step 3:**
- `rmse = np.sqrt(mean_squared_error(y_test, y_pred))`
- `mae  = mean_absolute_error(y_test, y_pred)`
- `r2   = r2_score(y_test, y_pred)`

</details>

In [ ]:
# GRADED FUNCTION: prepare_baseline_model
def prepare_baseline_model(X_train, X_test, y_train, y_test):
    """
    Train a baseline Linear Regression model and return performance metrics.

    Args:
        X_train (pd.DataFrame): Training features (complete, no missing values).
        X_test  (pd.DataFrame): Test features.
        y_train (pd.Series): Training target.
        y_test  (pd.Series): Test target.

    Returns:
        dict: {'rmse': float, 'mae': float, 'r2': float}
    """
    ### START CODE HERE ###

    model  = None
    # Fit the model
    None

    y_pred = None

    rmse = None
    mae  = None
    r2   = None

    ### END CODE HERE ###

    return {'rmse': rmse, 'mae': mae, 'r2': r2}

In [ ]:
X_train_c, X_test_c, y_train, y_test = train_test_split(
    X_complete, y, test_size=0.2, random_state=42
)

baseline = prepare_baseline_model(X_train_c, X_test_c, y_train, y_test)
print("Baseline (Complete Data):")
print(f"  RMSE: {baseline['rmse']:.4f}")
print(f"  MAE:  {baseline['mae']:.4f}")
print(f"  R2:   {baseline['r2']:.4f}")

In [ ]:
# Test your code!
unittests.exercise_1(prepare_baseline_model)

<a name='3---deletion-impact'></a>
## 3 - Deletion Impact

Listwise deletion is the baseline imputation strategy. The key question is: how much does losing rows reduce performance, and does the remaining sample remain representative?

<a name='exercise-2---apply_listwise_deletion'></a>
### **Exercise 2 - `apply_listwise_deletion`**

**Your Task:**

Implement `apply_listwise_deletion(X, y)` that drops rows where any feature is missing and aligns `y` accordingly.

**Requirements:**
- Return a tuple `(X_clean, y_clean, deleted_count)`
- `X_clean` must have no missing values
- `y_clean` must have the same length as `X_clean`
- `deleted_count` is an integer: the number of rows removed

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

**Step 1:** `complete_mask = X.notnull().all(axis=1)` — boolean mask for complete rows

**Step 2:** `X_clean = X[complete_mask]`, `y_clean = y[complete_mask]`

**Step 3:** `deleted_count = (~complete_mask).sum()`

</details>

In [ ]:
# GRADED FUNCTION: apply_listwise_deletion
def apply_listwise_deletion(X, y):
    """
    Drop rows with any missing feature and align the target vector.

    Args:
        X (pd.DataFrame): Feature matrix with potential missing values.
        y (pd.Series): Target vector aligned with X.

    Returns:
        tuple: (X_clean, y_clean, deleted_count)
    """
    ### START CODE HERE ###

    complete_mask  = None
    X_clean        = None
    y_clean        = None
    deleted_count  = None

    ### END CODE HERE ###

    return X_clean, y_clean, deleted_count

In [ ]:
X_clean, y_clean, n_deleted = apply_listwise_deletion(X_missing, y)
print(f"Original rows:  {len(X_missing)}")
print(f"Rows deleted:   {n_deleted}")
print(f"Remaining rows: {len(X_clean)}")
print(f"Missing values: {X_clean.isnull().sum().sum()}")

In [ ]:
# Test your code!
unittests.exercise_2(apply_listwise_deletion)

<a name='4---simple-imputation-impact'></a>
## 4 - Simple Imputation Impact

Unlike deletion, imputation preserves sample size but may introduce bias. Mean imputation is the most common but is known to **underestimate variance** and **distort correlations**.

<a name='exercise-3---apply_simple_imputation'></a>
### **Exercise 3 - `apply_simple_imputation`**

**Your Task:**

Implement `apply_simple_imputation(X_missing, strategy='mean')` that fits a `SimpleImputer` on the provided data and returns both the imputed DataFrame and the fitted imputer.

**Requirements:**
- Fit a `SimpleImputer(strategy=strategy)` on `X_missing`
- Return a tuple `(X_imputed, imputer)` where `X_imputed` is a `pd.DataFrame` with no missing values and `imputer` is the fitted `SimpleImputer` object

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
imputer = SimpleImputer(strategy=strategy)
imputer.fit(X_missing)
X_imputed = pd.DataFrame(
    imputer.transform(X_missing),
    columns=X_missing.columns,
    index=X_missing.index
)
return X_imputed, imputer
```

</details>

In [ ]:
# GRADED FUNCTION: apply_simple_imputation
def apply_simple_imputation(X_missing, strategy='mean'):
    """
    Fit a SimpleImputer on X_missing and return the imputed DataFrame plus the fitted imputer.

    Args:
        X_missing (pd.DataFrame): Feature matrix with missing values.
        strategy (str): 'mean' or 'median'.

    Returns:
        tuple: (X_imputed, imputer)
            - X_imputed (pd.DataFrame): Imputed copy with no missing values.
            - imputer (SimpleImputer): The fitted SimpleImputer object.
    """
    ### START CODE HERE ###

    imputer    = None
    X_imputed  = None

    ### END CODE HERE ###

    return X_imputed, imputer

In [ ]:
X_mean_imp, mean_imputer = apply_simple_imputation(X_missing, strategy='mean')
print(f"Missing after mean imputation:   {X_mean_imp.isnull().sum().sum()}")
print(f"Imputer type:                    {type(mean_imputer).__name__}")
print(f"Imputer statistics (first 3):   {mean_imputer.statistics_[:3] if mean_imputer is not None else 'N/A'}")

In [ ]:
# Test your code!
unittests.exercise_3(apply_simple_imputation)

<a name='5---advanced-imputation-impact'></a>
## 5 - Advanced Imputation Impact

Advanced methods (KNN, iterative) exploit multivariate structure and typically produce imputed values that are closer to the true values on MAR data, at the cost of higher computational overhead.

<a name='exercise-4---apply_advanced_imputation'></a>
### **Exercise 4 - `apply_advanced_imputation`**

**Your Task:**

Implement `apply_advanced_imputation(X_missing)` that applies **both** KNN and iterative imputation on the same dataset and returns both results.

**Requirements:**
- Apply `KNNImputer(n_neighbors=5)` and `IterativeImputer(max_iter=10, random_state=42)` independently
- Return a tuple `(X_knn, X_iterative)` — both `pd.DataFrames` with no missing values

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
knn_imputer  = KNNImputer(n_neighbors=5)
iter_imputer = IterativeImputer(max_iter=10, random_state=42)

X_knn = pd.DataFrame(knn_imputer.fit_transform(X_missing),
                     columns=X_missing.columns, index=X_missing.index)
X_iterative = pd.DataFrame(iter_imputer.fit_transform(X_missing),
                            columns=X_missing.columns, index=X_missing.index)
```

</details>

In [ ]:
# GRADED FUNCTION: apply_advanced_imputation
def apply_advanced_imputation(X_missing):
    """
    Apply both KNN and iterative imputation on the same dataset.

    Args:
        X_missing (pd.DataFrame): Feature matrix with missing values.

    Returns:
        tuple: (X_knn, X_iterative) — both pd.DataFrames with no missing values.
    """
    ### START CODE HERE ###

    X_knn        = None
    X_iterative  = None

    ### END CODE HERE ###

    return X_knn, X_iterative

In [ ]:
X_knn, X_iterative = apply_advanced_imputation(X_missing)
print(f"Missing after KNN imputation:        {X_knn.isnull().sum().sum() if X_knn is not None else 'N/A'}")
print(f"Missing after iterative imputation:  {X_iterative.isnull().sum().sum() if X_iterative is not None else 'N/A'}")

In [ ]:
# Test your code!
unittests.exercise_4(apply_advanced_imputation)

<a name='6---comprehensive-impact-report'></a>
## 6 - Comprehensive Impact Report

Bring all strategies together into a single comparison report that shows both model performance and distributional fidelity metrics.

<a name='exercise-5---create_impact_report'></a>
### **Exercise 5 - `create_impact_report`**

**Your Task:**

Implement `create_impact_report(results_dict, original_data, imputed_data)` that compares imputation strategies and identifies the best one.

**Requirements:**
- `results_dict`: a dict mapping strategy name → `{'rmse': ..., 'mae': ..., 'r2': ...}`
- `original_data`: a `pd.Series` of original (pre-missing) values
- `imputed_data`: a `pd.Series` of imputed values for comparison
- Return a **dict** with keys:
  - `'summary_table'` — `pd.DataFrame` summarising all strategies
  - `'best_strategy'` — name of the strategy with the lowest RMSE
  - `'distribution_metrics'` — dict with KS-test results comparing `original_data` vs `imputed_data`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand)</font></b></summary>

```python
from scipy import stats

summary_table = pd.DataFrame([

    {'strategy': k, **v} for k, v in results_dict.items()</details>

])

best_strategy = summary_table.loc[summary_table['rmse'].idxmin(), 'strategy']```

ks_stat, ks_p = stats.ks_2samp(original_data.dropna(), imputed_data.dropna())        'distribution_metrics': distribution_metrics}

distribution_metrics = {'ks_statistic': ks_stat, 'ks_p_value': ks_p}return {'summary_table': summary_table, 'best_strategy': best_strategy,

In [ ]:
# GRADED FUNCTION: create_impact_report
def create_impact_report(results_dict, original_data, imputed_data):
    """
    Compare imputation strategies and identify the best one, including distribution metrics.

    Args:
        results_dict (dict): Maps strategy name -> {'rmse': float, 'mae': float, 'r2': float}.
        original_data (pd.Series): Original values before missingness was introduced.
        imputed_data  (pd.Series): Imputed values for distribution comparison.

    Returns:
        dict with keys:
            'summary_table'       — pd.DataFrame with one row per strategy
            'best_strategy'       — str, strategy with lowest RMSE
            'distribution_metrics'— dict with 'ks_statistic' and 'ks_p_value'
    """
    ### START CODE HERE ###
    from scipy import stats

    summary_table        = None
    best_strategy        = None
    distribution_metrics = None

    ### END CODE HERE ###

    return {
        'summary_table': summary_table,
        'best_strategy': best_strategy,
        'distribution_metrics': distribution_metrics
    }

In [ ]:
# Build a sample results_dict using our previously computed baseline
results_dict_example = {
    'baseline':   prepare_baseline_model(*train_test_split(X_complete, y, test_size=0.2, random_state=42)),

    'mean_imp':   prepare_baseline_model(*train_test_split(X_complete, y, test_size=0.2, random_state=42)),    print(report.get('summary_table'))

}    print(f"Distribution metrics:  {report.get('distribution_metrics')}")

original_col  = X_complete.iloc[:, 0]    print(f"Best strategy:         {report.get('best_strategy')}")

imputed_col   = X_mean_imp.iloc[:, 0] if X_mean_imp is not None else original_colif report:

report = create_impact_report(results_dict_example, original_col, imputed_col)

In [ ]:
# Test your code!
unittests.exercise_5(create_impact_report)

In [ ]:
# Test your code!
unittests.exercise_5(create_impact_report)